<a href="https://colab.research.google.com/github/senchiao/HRRR_plots/blob/main/gfs_aws_200hpa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, timezone
import os

# Install basemap and its dependencies
!pip install basemap
!pip install basemap-data-hires
!pip install netCDF4

# Remove the problematic PROJ_LIB environment variable setting
# os.environ['PROJ_LIB']='/Users/schiao/miniconda3/lib/python3.7/site-packages/mpl_toolkits'

from mpl_toolkits.basemap import Basemap, addcyclic
from scipy.ndimage import minimum_filter, maximum_filter # Updated import
from netCDF4 import Dataset

In [2]:
!pip uninstall -y boto3 botocore
!pip install s3fs cartopy boto3==1.43.75 botocore==1.43.75 cfgrib
import s3fs
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from datetime import datetime, timedelta, timezone # Modified import
import boto3
from botocore import UNSIGNED
from botocore.config import Config

Found existing installation: boto3 1.43.75
Uninstalling boto3-1.43.75:
  Successfully uninstalled boto3-1.43.75
Found existing installation: botocore 1.43.75
Uninstalling botocore-1.43.75:
  Successfully uninstalled botocore-1.43.75
  Using cached boto3-1.43.75-py3-none-any.whl.metadata (6.6 kB)
  Using cached botocore-1.43.75-py3-none-any.whl.metadata (5.6 kB)
Using cached boto3-1.43.75-py3-none-any.whl (140 kB)
Using cached botocore-1.43.75-py3-none-any.whl (15.7 MB)


In [3]:
def plot_gfs_200hpa_wind(date, cycle, forecast_hour, region=None):
    """
    Reads GFS 200 hPa wind data from AWS and plots it.

    Args:
        date (datetime.date): The date of the GFS model run.
        cycle (int): The forecast cycle (00, 06, 12, or 18 UTC).
        forecast_hour (int): The forecast hour (e.g., 0, 6, 12, etc.).
        region (list, optional): A list [lon_min, lon_max, lat_min, lat_max]
                                  to define a specific plotting region. Defaults to None (global).
    """
    print(f"Starting plot_gfs_200hpa_wind for date={date}, cycle={cycle}, forecast_hour={forecast_hour}")

    # Initialize S3 file system
    fs = s3fs.S3FileSystem(anon=True) # anon=True for public buckets

    # Construct the S3 path for a GRIB2 file
    # For 0.25 degree resolution data:
    # Path: s3://noaa-gfs-bdp-pds/gfs.YYYYMMDD/CC/atmos/gfs.tCCz.pgrb2.0p25.fHHH.grib2
    s3_path = (f"s3://noaa-gfs-bdp-pds/gfs.{date.strftime('%Y%m%d')}/"
               f"{str(cycle).zfill(2)}/atmos/gfs.t{str(cycle).zfill(2)}z.pgrb2.0p25.f{str(forecast_hour).zfill(3)}.grib2")

    print(f"Attempting to open GFS data from: {s3_path}")

    try:
        with fs.open(s3_path, 'rb') as f:
            # Use xarray with cfgrib engine to open the GRIB2 file
            # We need U and V components of wind at 200 hPa (isobaricInhPa level)
            # 'filter_by_keys' is crucial to select specific levels/variables efficiently
            ds = xr.open_dataset(f, engine="cfgrib",
                                 backend_kwargs={'filter_by_keys': {'typeOfLevel': 'isobaricInhPa', 'level': 200}})

            print("\nDataset loaded. Variables found at 200 hPa:")
            print(list(ds.keys()))

            # Check if U and V wind components are available
            if 'u' in ds and 'v' in ds:
                u_wind = ds['u'].squeeze()
                v_wind = ds['v'].squeeze()

                # Calculate wind speed for contouring (optional, but good for visualizing jet streams)
                wind_speed = np.sqrt(u_wind**2 + v_wind**2)

                # Set up the plot
                fig = plt.figure(figsize=(12, 8))
                ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

                if region:
                    ax.set_extent(region, crs=ccrs.PlateCarree())
                else:
                    ax.set_global()

                ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
                ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
                ax.add_feature(cfeature.STATES, linewidth=0.3, edgecolor='gray') # For CONUS plots

                # Add gridlines with labels
                gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                                  linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
                gl.top_labels = False
                gl.right_labels = False

                # Plot wind speed as contours (optional)
                clevs = np.arange(10, 101, 10) # Wind speed contours from 10 to 100 m/s
                cbar_contour = ax.contourf(ds.longitude, ds.latitude, wind_speed,
                                           levels=clevs, cmap='viridis', extend='max', transform=ccrs.PlateCarree())
                plt.colorbar(cbar_contour, ax=ax, orientation='vertical', label='Wind Speed (m/s)')


                # Plot wind vectors (quivers or barbs)
                # To avoid plotting too many arrows, we can thin the data
                skip = 10 # Plot every 10th data point
                ax.quiver(ds.longitude[::skip], ds.latitude[::skip],
                          u_wind[::skip, ::skip], v_wind[::skip, ::skip],
                          color='black', transform=ccrs.PlateCarree(),
                          scale=1000, width=0.002, headwidth=3, headlength=4, alpha=0.7)

                ax.set_title(f'GFS 200 hPa Wind Forecast\n'
                             f'Run: {date.strftime("%Y-%m-%d")} {str(cycle).zfill(2)}Z, '
                             f'Forecast: +{str(forecast_hour).zfill(3)}h')

                plt.tight_layout()
                plt.show()

            else:
                print("Error: 'u' and 'v' wind components not found at 200 hPa.")

    except FileNotFoundError:
        print(f"Error: GFS file not found at {s3_path}")
        return
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return
    finally:
        # Ensure the S3 filesystem object is closed if it's no longer needed
        # In this case, s3fs might manage connections, but explicitly closing can be good practice.
        # This block will always execute.
        print("Finished attempting to load GFS data.")


In [4]:
# The plotting logic has been moved inside the 'plot_gfs_200hpa_wind' function in cell 'pzyYuQR_rxbK'.

In [5]:
from datetime import datetime, timedelta, timezone # Added to ensure datetime is defined
import numpy as np # Added to ensure numpy is defined for extrema
from scipy.ndimage import minimum_filter, maximum_filter # Added for extrema function
import boto3 # Added for client
from botocore import UNSIGNED # Added for client
from botocore.config import Config # Added for client

def extrema(mat,mode='wrap',window=10):
    """find the indices of local extrema (min and max)
    in the input array."""
    mn = minimum_filter(mat, size=window, mode=mode)
    mx = maximum_filter(mat, size=window, mode=mode)
    # (mat == mx) true if pixel is equal to the local max
    # (mat == mn) true if pixel is equal to the local in
    # Return the indices of the maxima, minima
    return np.nonzero(mat == mn), np.nonzero(mat == mx)

# plot 00 UTC today or previous days if data is not available.
for i in range(5): # Try up to 5 previous days
    target_date = datetime.now(timezone.utc) - timedelta(days=i)
    date_str = target_date.strftime('%Y%m%d')
    try:
        # Connect anonymously to the public bucket
        client = boto3.client('s3', config=Config(signature_version=UNSIGNED))
        # Construct the object name for the GFS file at 12Z, forecast 000
        s3_object_name = f"gfs.{date_str}/12/atmos/gfs.t12z.pgrb2.0p25.f000"
        # Download a specific GFS file (e.g., 0.5 resolution forecast)
        client.download_file('noaa-gfs-bdp-pds', s3_object_name, 'gfs.t12z.pgrb2.0p25.f000')

        # open OpenDAP dataset.
        #data=Dataset("https://nomads.ncep.noaa.gov:9090/dods/gfs_0p25/gfs%s/gfs_0p25_%sz_anl" %\
         #      (date_str[0:8],date_str[8:10]))

        date = date_str # Assign the successful date string to 'date' for title.
        print(f"Successfully retrieved data for date: {date_str}")
        break # Exit loop if data is successfully retrieved
    except client.exceptions.NoSuchKey:
        print(f"GFS file not found on S3 for date: {date_str}. Trying previous day...")
    except Exception as e:
        print(f"Failed to retrieve data for date: {date_str}. Error: {e}. Trying previous day...")
else:
    raise OSError("Could not retrieve GFS data for the last 5 days.")

Failed to retrieve data for date: 20260908. Error: An error occurred (404) when calling the HeadObject operation: Not Found. Trying previous day...
Successfully retrieved data for date: 20260907


In [6]:
# read lats,lons.
lats = data.variables['lat'][:]
lons1 = data.variables['lon'][:]
nlats = len(lats)
nlons = len(lons1)
# read prmsl, convert to hPa (mb).
prmsl = 0.01*data.variables['prmslmsl'][0]
# the window parameter controls the number of highs and lows detected.
# (higher value, fewer highs and lows)
local_min, local_max = extrema(prmsl, mode='wrap', window=50)
# create Basemap instance. (change the area)
#
m =\
Basemap(llcrnrlon=210,llcrnrlat=15,urcrnrlon=280,urcrnrlat=45,projection='mill')
# add wrap-around point in longitude.
prmsl, lons = addcyclic(prmsl, lons1)
# contour levels
clevs = np.arange(900,1100.,3.)
# find x,y of map projection grid.
lons, lats = np.meshgrid(lons, lats)
x, y = m(lons, lats)
# create figure.
fig=plt.figure(figsize=(8,4.5))
ax = fig.add_axes([0.05,0.05,0.9,0.85])
cs = m.contour(x,y,prmsl,clevs,colors='k',linewidths=1.)
m.drawcoastlines(linewidth=1.25)
m.fillcontinents(color='0.8')
m.drawparallels(np.arange(-80,81,10),labels=[1,1,0,0])
m.drawmeridians(np.arange(0,360,30),labels=[0,0,0,1])
xlows = x[local_min]; xhighs = x[local_max]
ylows = y[local_min]; yhighs = y[local_max]
lowvals = prmsl[local_min]; highvals = prmsl[local_max]
# plot lows as blue L's, with min pressure value underneath.
xyplotted = []
# don't plot if there is already a L or H within dmin meters.
yoffset = 0.022*(m.ymax-m.ymin)
dmin = yoffset
for x,y,p in zip(xlows, ylows, lowvals):
    if x < m.xmax and x > m.xmin and y < m.ymax and y > m.ymin:
        dist = [np.sqrt((x-x0)**2+(y-y0)**2) for x0,y0 in xyplotted]
        if not dist or min(dist) > dmin:
            plt.text(x,y,'L',fontsize=14,fontweight='bold',
                    ha='center',va='center',color='b')
            plt.text(x,y-yoffset,repr(int(p)),fontsize=9,
                    ha='center',va='top',color='b',
                    bbox = dict(boxstyle="square",ec='None',fc=(1,1,1,0.5)))
            xyplotted.append((x,y))
# plot highs as red H's, with max pressure value underneath.
xyplotted = []
for x,y,p in zip(xhighs, yhighs, highvals):
    if x < m.xmax and x > m.xmin and y < m.ymax and y > m.ymin:
        dist = [np.sqrt((x-x0)**2+(y-y0)**2) for x0,y0 in xyplotted]
        if not dist or min(dist) > dmin:
            plt.text(x,y,'H',fontsize=14,fontweight='bold',
                    ha='center',va='center',color='r')
            plt.text(x,y-yoffset,repr(int(p)),fontsize=9,
                    ha='center',va='top',color='r',
                    bbox = dict(boxstyle="square",ec='None',fc=(1,1,1,0.5)))
            xyplotted.append((x,y))
plt.title('Mean Sea-Level Pressure (with Highs and Lows) %s' % date)
plt.show()

NameError: name 'data' is not defined

In [ ]:
import xarray as xr

# Open the downloaded GRIB2 file using xarray and cfgrib engine
# Using filter_by_keys to select a specific typeOfLevel and level as suggested by the error
ds_local = xr.open_dataset('gfs.t12z.pgrb2.0p25.f000', engine='cfgrib', backend_kwargs={'filter_by_keys': {'typeOfLevel': 'isobaricInhPa', 'level': 200}})

# Display the dataset information
display(ds_local)

if __name__ == "__main__":
    # Define the date and forecast
    current_date = datetime.date(2026, 8, 4) # YYYY, M, D
    current_cycle = 12 # 00, 06, 12, or 18 UTC
    forecast_time = 0 # Forecast hour, e.g., 0 for analysis, 6 for 6-hour forecast

plot_gfs_200hpa_wind(current_date, current_cycle, forecast_time)